In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import cluster
import numpy as np

In [ ]:
data = gpd.read_parquet("/data/uscuni-restricted/geodem/05_data/_data_f_m.parquet")

In [ ]:
# prg = data[data["naz_kraj"]=="Hlavní město Praha"]

In [ ]:
data = data.drop(
    columns=["Obyvatelstvo celkem", "Hospodařící domácnosti v bytech celkem"]
)

In [ ]:
fa = data.columns[13:]

In [ ]:
fa = [
    "Obyvatelstvo - věk: 0 - 14  - celkem",
    "Obyvatelstvo - věk: 15 - 39  - celkem",
    "Obyvatelstvo - věk: 40 - 64  - celkem",
    "Obyvatelstvo - věk: 65 a více - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: ve vlastním domě",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Počet obyvatel na byt",
    "Počet osob v bytech celkem  - se 3 obytnými místnostmi",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem",
    "Obyvatelstvo - ekon. aktivita: zaměstnaní - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby v domácnosti, děti předškolního věku, ostatní závislé osoby - celkem",
    "Obyvatelstvo - ekon. aktivita: žáci, studenti - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání:  vysokoškolské - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem",
    "Zaměstnaní - Pracovníci ve službách a prodeji",
    "Zaměstnaní - Řemeslníci a opraváři",
    "Zaměstnaní - nezjištěno",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem",
    "Hospodařící domácnosti v bytech - typ HD: rodinné domácnosti tvořené 1 rodinou - úplná rodina bez závislých dětí",
    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem",
    "Obyvatelstvo - náboženská víra: věřící - hlásící se k církvi, náboženské společnosti nebo směru - celkem",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem",
    "Obyvatelstvo - s trvalým pobytem - celkem",
    "Obyvatelstvo - bydlící v bytech - celkem",
    "Obyvatelstvo - bydlící v zařízeních - celkem",
]

In [ ]:
inertias = {}

for k in range(2, 50):
    kmeans = cluster.KMeans(n_clusters=k, random_state=42)
    kmeans.fit(data[fa])
    inertias[k] = kmeans.inertia_

In [ ]:
# Assuming 'inertias' is a dictionary or list where keys/index are your N values
inertia_series = pd.Series(inertias)

plt.figure(figsize=(20, 10))
ax = inertia_series.plot(marker="o", linestyle="-", color="b")

# This is the "all the ticks" magic:
# We tell the axis to use every single value in the index
plt.xticks(inertia_series.index)

plt.title("Inertia (WCSS) across PC Iterations", fontsize=16)
plt.xlabel("Number of PCs (N)", fontsize=12)
plt.ylabel("Inertia", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.7)

plt.show()

In [ ]:
from clustergram import Clustergram

cgram = Clustergram(range(1, 30))
cgram.fit(data[fa])

In [ ]:
cgram

In [ ]:
# 1. Create the plot
ax = cgram.plot(figsize=(20, 30))
ax.set_xticks(np.arange(1, 31))
ax.grid(True, which="both", color="black", linestyle="-", linewidth=0.5)

In [ ]:
df = data[fa]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cluster_col = "kmeans_fa"
var_cols = [c for c in fa if c != cluster_col]  # fa list minus 'kmeans_fa'

short_names = [
    "Age 0–14",
    "Age 15–39",
    "Age 40–64",
    "Age 65+",
    "Owner-occupied (house)",
    "Owner-occupied (flat)",
    "Persons per dwelling",
    "3-room dwellings",
    "Slovak citizenship",
    "EU citizenship",
    "Unknown citizenship",
    "Employed",
    "Homemakers & dependants",
    "Students",
    "Secondary (no matura)",
    "Secondary (matura)",
    "University degree",
    "Education unknown",
    "Service & sales workers",
    "Craft & repair workers",
    "Occupation unknown",
    "Agriculture sector",
    "Industry sector",
    "Couple family (no children)",
    "Married",
    "Religious (affiliated)",
    "No religion",
    "Religion not stated",
    "Permanent residents",
    "Living in dwellings",
    "Living in institutions",
]

cluster_means = df.groupby(cluster_col)[var_cols].mean()

N = len(var_cols)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

colors = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(11, 11), subplot_kw=dict(polar=True))

for i, (cluster_id, row) in enumerate(cluster_means.iterrows()):
    vals = row.tolist() + row.tolist()[:1]
    color = colors[i % len(colors)]
    ax.plot(angles, vals, linewidth=2, label=f"Cluster {cluster_id}", color=color)
    ax.fill(angles, vals, alpha=0.07, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(short_names, fontsize=8)
ax.set_title("Geodemographic Cluster Profiles (z-scores)", fontsize=14, pad=25)
ax.set_rlabel_position(0)
ax.yaxis.set_tick_params(labelsize=8)
ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.15), fontsize=10)

plt.tight_layout()
plt.savefig("spider_chart.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
kmeans5 = cluster.KMeans(n_clusters=3, random_state=42)

In [ ]:
kmeans5.fit(data[fa])

In [ ]:
k = 3
labels = kmeans5.labels_

# Sort cluster centers by the first feature
sorted_centers_indices = np.argsort(kmeans5.cluster_centers_[:, 0])

# Reassign labels: the cluster with the smallest center gets label 0, etc.
new_labels = np.zeros_like(labels)
for i, cluster_idx in enumerate(sorted_centers_indices):
    new_labels[labels == cluster_idx] = i

data["kmeans_fa"] = new_labels

In [ ]:
data[["kmeans_fa", "geometry"]].plot("kmeans_fa", categorical=True, cmap="tab20")

# PCA

In [ ]:
# prg = data[data["naz_kraj"]=="Hlavní město Praha"]

In [ ]:
pca = data.columns[13:]

In [ ]:
pca = [
    "Obyvatelstvo - věk: 0 - 14  - celkem",
    "Obyvatelstvo - věk: 15 - 39  - celkem",
    "Obyvatelstvo - věk: 40 - 64  - celkem",
    "Obyvatelstvo - věk: 65 a více - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: ve vlastním domě",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Počet obyvatel na byt",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem",
    "Obyvatelstvo - ekon. aktivita: zaměstnaní - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby v domácnosti, děti předškolního věku, ostatní závislé osoby - celkem",
    "Obyvatelstvo - ekon. aktivita: žáci, studenti - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání:  vysokoškolské - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem",
    "Zaměstnaní - Pracovníci ve službách a prodeji",
    "Zaměstnaní - Kvalifikovaní pracovníci v zemědělství,lesnictví a rybářství",
    "Zaměstnaní - Řemeslníci a opraváři",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: osoby pracující na vlastní účet - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: stavebnictví - celkem",
    "Hospodařící domácnosti v bytech - typ HD: domácnosti jednotlivců",
    "Hospodařící domácnosti v bytech - typ HD: rodinné domácnosti tvořené 1 rodinou - úplná rodina bez závislých dětí",
    "Hospodařící domácnosti v bytech - typ HD: rodinné domácnosti tvořené 1 rodinou - úplná rodina se závislými dětmi",
    "Obyvatelstvo - rodinný stav: svobodní  - celkem",
    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem",
    "Obyvatelstvo - rodinný stav: ovdovělí - celkem",
    "Obyvatelstvo - náboženská víra: věřící - hlásící se k církvi, náboženské společnosti nebo směru - celkem",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem",
    "Obyvatelstvo - s trvalým pobytem - celkem",
    "Obyvatelstvo - bydlící v bytech - celkem",
    "Obyvatelstvo - bydlící v zařízeních - celkem",
]

In [ ]:
inertias = {}

for k in range(2, 50):
    kmeans = cluster.KMeans(n_clusters=k, random_state=42)
    kmeans.fit(data[pca])
    inertias[k] = kmeans.inertia_

In [ ]:
import matplotlib.pyplot as plt

# Assuming 'inertias' is a dictionary or list where keys/index are your N values
inertia_series = pd.Series(inertias)

plt.figure(figsize=(20, 10))
ax = inertia_series.plot(marker="o", linestyle="-", color="b")

# This is the "all the ticks" magic:
# We tell the axis to use every single value in the index
plt.xticks(inertia_series.index)

plt.title("Inertia (WCSS) across PC Iterations", fontsize=16)
plt.xlabel("Number of PCs (N)", fontsize=12)
plt.ylabel("Inertia", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.7)

plt.show()

In [ ]:
from clustergram import Clustergram

cgram = Clustergram(range(1, 30))
cgram.fit(data[pca])

In [ ]:
# 1. Create the plot
ax = cgram.plot(figsize=(20, 30))
ax.set_xticks(np.arange(1, 31))
ax.grid(True, which="both", color="black", linestyle="-", linewidth=0.5)

In [ ]:
kmeans5 = cluster.KMeans(n_clusters=4, random_state=42)

In [ ]:
kmeans5.fit(data[pca])

In [ ]:
k = 4
labels = kmeans5.labels_

# Sort cluster centers by the first feature
sorted_centers_indices = np.argsort(kmeans5.cluster_centers_[:, 0])

# Reassign labels: the cluster with the smallest center gets label 0, etc.
new_labels = np.zeros_like(labels)
for i, cluster_idx in enumerate(sorted_centers_indices):
    new_labels[labels == cluster_idx] = i

data["kmeans_pca"] = new_labels

In [ ]:
data[["kmeans_pca", "geometry"]].explore("kmeans_pca", categorical=True, cmap="tab20")

In [ ]:
data[["kmeans_fa", "geometry"]].plot("kmeans_fa", categorical=True, cmap="tab20")

In [ ]:
k5_means = data.groupby("kmeans_pca")[pca].mean()
k5_means.T.style.background_gradient(cmap="coolwarm", vmin=-5, vmax=5, axis=1)